# Video-only RVQ-VAE: TF-IDF và temporal token models

Notebook dùng duy nhất đặc trưng ViTPose của bài báo. Nó so sánh TF-IDF với embedding + 1D CNN và embedding + bidirectional GRU. Chạy `smoke` trước, sau đó đổi thành `full` để đánh giá đủ 9 fold.

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/xndung/fidgety/data")
REPO_URL = "https://github.com/xn-dung/Fidgety.git"
SOURCE_DIR = Path("/kaggle/working/Fidgety")
UPLOADED_PROJECT_DIR = None
RUN_MODE = "smoke"
SEED = 42
GPU_WORKERS = 2
CPU_THREADS_PER_WORKER = 2
OUTPUT_DIR = Path(f"/kaggle/working/video_rvq_{RUN_MODE}")

if RUN_MODE == "smoke":
    FOLDS = [1]
    EPOCHS = 5
    PATIENCE = 3
elif RUN_MODE == "full":
    FOLDS = list(range(1, 10))
    EPOCHS = 80
    PATIENCE = 10
else:
    raise ValueError("RUN_MODE must be smoke or full")

## Định nghĩa temporal token classifiers

Hai mô hình giữ nguyên thứ tự 25 token: embedding khởi tạo từ codebook RVQ kết hợp dilated 1D CNN, và embedding kết hợp bidirectional GRU với attention. Epoch và threshold được chọn trên validation set của từng fold.

In [ ]:
TEMPORAL_SOURCE = r'''
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
from queue import Empty, Queue
import json
import threading

from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from data import TextStore, load_video_fold
from model import RVQVAE, encode_tokens

TEMPORAL_EPOCHS = 20 if RUN_MODE == "smoke" else 120
TEMPORAL_PATIENCE = 5 if RUN_MODE == "smoke" else 15
TEMPORAL_BATCH_SIZE = 64

class MotionTokenEmbedding(nn.Module):
    def __init__(self, codebooks, steps=25, dropout=0.2):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding.from_pretrained(weight.detach().clone(), freeze=False) for weight in codebooks])
        dimension = codebooks[0].shape[1]
        self.position = nn.Parameter(torch.zeros(1, steps, dimension))
        nn.init.normal_(self.position, std=0.02)
        self.norm = nn.LayerNorm(dimension)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tokens):
        values = torch.stack([embedding(tokens[:, :, level]) for level, embedding in enumerate(self.embeddings)]).sum(0)
        return self.dropout(self.norm(values + self.position[:, :values.shape[1]]))

class TemporalResidualBlock(nn.Module):
    def __init__(self, channels, dilation, dropout):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(channels)
        )
        self.activation = nn.GELU()

    def forward(self, values):
        return self.activation(values + self.layers(values))

class TokenCNN(nn.Module):
    def __init__(self, codebooks):
        super().__init__()
        dimension = codebooks[0].shape[1]
        self.embedding = MotionTokenEmbedding(codebooks)
        self.input = nn.Sequential(nn.Conv1d(dimension, 128, 1), nn.BatchNorm1d(128), nn.GELU())
        self.blocks = nn.Sequential(*[TemporalResidualBlock(128, dilation, 0.25) for dilation in (1, 2, 4)])
        self.output = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.35), nn.Linear(128, 1))

    def forward(self, tokens):
        values = self.blocks(self.input(self.embedding(tokens).transpose(1, 2)))
        pooled = torch.cat([values.mean(-1), values.amax(-1)], dim=1)
        return self.output(pooled).squeeze(1)

class TokenGRU(nn.Module):
    def __init__(self, codebooks):
        super().__init__()
        dimension = codebooks[0].shape[1]
        self.embedding = MotionTokenEmbedding(codebooks)
        self.gru = nn.GRU(dimension, 96, num_layers=2, batch_first=True, bidirectional=True, dropout=0.25)
        self.attention = nn.Linear(192, 1)
        self.output = nn.Sequential(nn.Linear(384, 128), nn.GELU(), nn.Dropout(0.35), nn.Linear(128, 1))

    def forward(self, tokens):
        values, _ = self.gru(self.embedding(tokens))
        weights = torch.softmax(self.attention(values).squeeze(-1), dim=1).unsqueeze(-1)
        pooled = torch.cat([(values * weights).sum(1), values.amax(1)], dim=1)
        return self.output(pooled).squeeze(1)

def temporal_metrics(labels, probabilities, threshold):
    labels = np.asarray(labels, dtype=np.int64)
    predictions = (np.asarray(probabilities) >= threshold).astype(np.int64)
    tn = int(np.sum((labels == 0) & (predictions == 0)))
    fp = int(np.sum((labels == 0) & (predictions == 1)))
    fn = int(np.sum((labels == 1) & (predictions == 0)))
    tp = int(np.sum((labels == 1) & (predictions == 1)))
    sensitivity = tp / (tp + fn) if tp + fn else float("nan")
    specificity = tn / (tn + fp) if tn + fp else float("nan")
    return {"sensitivity": sensitivity, "specificity": specificity, "balanced_accuracy": (sensitivity + specificity) / 2, "tn": tn, "fp": fp, "fn": fn, "tp": tp}, predictions

def select_threshold(labels, probabilities):
    candidates = np.linspace(0.05, 0.95, 181)
    scores = [temporal_metrics(labels, probabilities, threshold)[0]["balanced_accuracy"] for threshold in candidates]
    best = np.flatnonzero(np.isclose(scores, np.max(scores)))
    index = int(best[np.argmin(np.abs(candidates[best] - 0.5))])
    return float(candidates[index]), float(scores[index])

def token_loader(tokens, labels, batch_size, shuffle, seed):
    generator = torch.Generator().manual_seed(seed)
    dataset = TensorDataset(torch.as_tensor(tokens, dtype=torch.long), torch.as_tensor(labels, dtype=torch.float32))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, generator=generator, pin_memory=True)

@torch.no_grad()
def temporal_probabilities(model, tokens, device):
    model.eval()
    loader = DataLoader(TensorDataset(torch.as_tensor(tokens, dtype=torch.long)), batch_size=128, shuffle=False, pin_memory=True)
    values = []
    for (batch,) in loader:
        values.append(torch.sigmoid(model(batch.to(device, non_blocking=True))).cpu().numpy())
    return np.concatenate(values)

model_initialization_lock = threading.Lock()

def train_temporal_model(name, codebooks, train_tokens, train_labels, val_tokens, val_labels, test_tokens, test_labels, device, seed):
    with model_initialization_lock:
        torch.manual_seed(seed)
        model = TokenCNN(codebooks) if name == "token_1d_cnn" else TokenGRU(codebooks)
    model.to(device)
    torch.cuda.set_device(device)
    torch.cuda.manual_seed(seed)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    negatives = float(np.sum(np.asarray(train_labels) == 0))
    positives = float(np.sum(np.asarray(train_labels) == 1))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(negatives / positives, device=device))
    loader = token_loader(train_tokens, train_labels, TEMPORAL_BATCH_SIZE, True, seed)
    scaler = torch.cuda.amp.GradScaler()
    best_score = -1.0
    best_state = deepcopy(model.state_dict())
    best_threshold = 0.5
    best_epoch = 0
    stale = 0
    for epoch in range(1, TEMPORAL_EPOCHS + 1):
        model.train()
        for tokens, labels in loader:
            tokens = tokens.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                loss = criterion(model(tokens), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        validation_probabilities = temporal_probabilities(model, val_tokens, device)
        threshold, score = select_threshold(val_labels, validation_probabilities)
        if score > best_score + 1e-6:
            best_score = score
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_threshold = threshold
            best_epoch = epoch
            stale = 0
        else:
            stale += 1
            if stale >= TEMPORAL_PATIENCE:
                break
    model.load_state_dict(best_state)
    test_probabilities = temporal_probabilities(model, test_tokens, device)
    metrics, predictions = temporal_metrics(test_labels, test_probabilities, best_threshold)
    return model, metrics, predictions, test_probabilities, best_threshold, best_epoch, best_score

def run_temporal_fold(fold, gpu):
    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    cached_rows = []
    for name in ("token_1d_cnn", "token_gru"):
        result_path = fold_dir / f"{name}_result.json"
        if result_path.is_file():
            cached_rows.append(json.loads(result_path.read_text()))
    if len(cached_rows) == 2:
        print(f"GPU {gpu} temporal fold {fold} skipped", flush=True)
        return cached_rows
    device = torch.device(f"cuda:{gpu}")
    torch.cuda.set_device(device)
    data = load_video_fold(TextStore(DATA_ROOT), fold)
    checkpoint = torch.load(fold_dir / "rvqvae.pt", map_location=device)
    configuration = checkpoint["config"]
    rvq = RVQVAE(patch_size=configuration["patch_size"], hidden_dim=configuration["hidden_dim"], latent_dim=configuration["latent_dim"], codebook_size=configuration["codebook_size"], quantizers=configuration["quantizers"], commitment_weight=configuration["commitment_weight"]).to(device)
    rvq.load_state_dict(checkpoint["state_dict"])
    codebooks = [codebook.weight.detach().clone() for codebook in rvq.quantizer.codebooks]
    train_tokens = encode_tokens(rvq, data.x_train, device, 128)
    test_tokens = encode_tokens(rvq, data.x_test, device, 128)
    split = len(data.y_fit)
    rows = []
    for offset, name in enumerate(("token_1d_cnn", "token_gru")):
        model, metrics, predictions, probabilities, threshold, epoch, validation_score = train_temporal_model(name, codebooks, train_tokens[:split], data.y_fit, train_tokens[split:], data.y_val, test_tokens, data.y_test, device, SEED + fold * 10 + offset)
        row = {"fold": fold, "classifier": name, "train_samples": len(data.y_fit), "test_samples": len(data.y_test), "vocabulary_size": configuration["codebook_size"] * configuration["quantizers"], **metrics}
        torch.save({"state_dict": model.state_dict(), "threshold": threshold, "best_epoch": epoch, "validation_balanced_accuracy": validation_score}, fold_dir / f"{name}.pt")
        pd.DataFrame({"y_true": data.y_test, "probability": probabilities, "y_pred": predictions}).to_csv(fold_dir / f"{name}_predictions.csv", index=False)
        (fold_dir / f"{name}_result.json").write_text(json.dumps(row, indent=2))
        rows.append(row)
        print(f"GPU {gpu} fold={fold} classifier={name} BA={metrics['balanced_accuracy']:.4f} val_BA={validation_score:.4f} threshold={threshold:.3f} epoch={epoch}", flush=True)
        del model
        torch.cuda.empty_cache()
    del rvq
    torch.cuda.empty_cache()
    return rows

temporal_queue = Queue()
for fold in FOLDS:
    temporal_queue.put(fold)

def temporal_gpu_worker(gpu):
    rows = []
    while True:
        try:
            fold = temporal_queue.get_nowait()
        except Empty:
            return rows
        rows.extend(run_temporal_fold(fold, gpu))
        temporal_queue.task_done()

with ThreadPoolExecutor(max_workers=worker_count) as executor:
    futures = [executor.submit(temporal_gpu_worker, gpu) for gpu in range(worker_count)]
    temporal_rows = [row for future in futures for row in future.result()]

base_results = pd.read_csv(OUTPUT_DIR / "fold_results.csv")
base_results = base_results[~base_results["classifier"].isin(["token_1d_cnn", "token_gru"])]
fold_results = pd.concat([base_results, pd.DataFrame(temporal_rows)], ignore_index=True).sort_values(["classifier", "fold"])
summary = summarize(fold_results)
fold_results.to_csv(OUTPUT_DIR / "fold_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
print(summary.to_string(index=False))
'''

## Chuẩn bị source

Nếu chưa đưa thư mục `đồ án` lên repository, hãy upload nó thành một Kaggle Dataset và đặt `UPLOADED_PROJECT_DIR` ở cell trên thành đường dẫn tới thư mục đó.

In [ ]:
import shutil
import subprocess
import sys

if UPLOADED_PROJECT_DIR is not None:
    PROJECT_DIR = Path(UPLOADED_PROJECT_DIR)
else:
    if SOURCE_DIR.exists():
        subprocess.run(["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(SOURCE_DIR)], check=True)
    PROJECT_DIR = SOURCE_DIR / "đồ án"

if not (PROJECT_DIR / "run.py").is_file():
    raise FileNotFoundError(f"Không tìm thấy project tại {PROJECT_DIR}. Hãy push source lên GitHub hoặc đặt UPLOADED_PROJECT_DIR.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
print("Project:", PROJECT_DIR)

In [ ]:
import torch

if not DATA_ROOT.is_dir() and (DATA_ROOT.parent / "data.zip").is_file():
    DATA_ROOT = DATA_ROOT.parent / "data.zip"
if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Settings > Accelerator")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_ROOT)
VISIBLE_GPUS = torch.cuda.device_count()
for gpu in range(VISIBLE_GPUS):
    print(f"GPU {gpu}:", torch.cuda.get_device_name(gpu))
print("Parallel GPU workers:", min(GPU_WORKERS, VISIBLE_GPUS, len(FOLDS)))
print("Folds:", FOLDS)
print("Epochs:", EPOCHS)

## Huấn luyện và đánh giá

In [ ]:
from collections import deque
import os
import time
import pandas as pd

sys.path.insert(0, str(PROJECT_DIR))
from experiment import summarize

JOBS_DIR = OUTPUT_DIR / "_jobs"
JOBS_DIR.mkdir(parents=True, exist_ok=True)

def fold_complete(fold):
    path = JOBS_DIR / f"fold_{fold}" / "fold_results.csv"
    if not path.is_file():
        return False
    frame = pd.read_csv(path)
    actual = set(zip(frame["fold"], frame["classifier"]))
    return {(fold, "tfidf_logistic"), (fold, "tfidf_mlp")}.issubset(actual)

def fold_command(fold):
    return [
        sys.executable, "-u", str(PROJECT_DIR / "run.py"),
        "--data-path", str(DATA_ROOT),
        "--output-dir", str(JOBS_DIR / f"fold_{fold}"),
        "--folds", str(fold),
        "--seed", str(SEED),
        "--epochs", str(EPOCHS),
        "--patience", str(PATIENCE),
        "--batch-size", "64",
        "--patch-size", "10",
        "--hidden-dim", "256",
        "--latent-dim", "64",
        "--codebook-size", "64",
        "--quantizers", "2",
        "--ngram-order", "2",
        "--device", "cuda"
    ]

worker_count = min(GPU_WORKERS, VISIBLE_GPUS, len(FOLDS))
pending = deque(fold for fold in FOLDS if not fold_complete(fold))
active = {}
print(f"Starting {worker_count} GPU workers for folds {FOLDS}")
try:
    while pending or active:
        for gpu in [index for index in range(worker_count) if index not in active]:
            if not pending:
                break
            fold = pending.popleft()
            environment = os.environ.copy()
            environment["CUDA_VISIBLE_DEVICES"] = str(gpu)
            environment["OMP_NUM_THREADS"] = str(CPU_THREADS_PER_WORKER)
            environment["MKL_NUM_THREADS"] = str(CPU_THREADS_PER_WORKER)
            process = subprocess.Popen(fold_command(fold), env=environment)
            active[gpu] = (process, fold)
            print(f"GPU {gpu} started fold {fold}, pid={process.pid}", flush=True)
        if active:
            time.sleep(1)
        for gpu, (process, fold) in list(active.items()):
            returncode = process.poll()
            if returncode is None:
                continue
            if returncode:
                raise RuntimeError(f"Fold {fold} failed on GPU {gpu} with code {returncode}")
            print(f"GPU {gpu} completed fold {fold}", flush=True)
            del active[gpu]
except BaseException:
    for process, _ in active.values():
        if process.poll() is None:
            process.terminate()
    for process, _ in active.values():
        process.wait()
    raise

frames = []
for fold in FOLDS:
    job = JOBS_DIR / f"fold_{fold}"
    frame = pd.read_csv(job / "fold_results.csv")
    frames.append(frame[frame["fold"] == fold])
    shutil.copytree(job / f"fold_{fold}", OUTPUT_DIR / f"fold_{fold}", dirs_exist_ok=True)
fold_results = pd.concat(frames, ignore_index=True).sort_values(["classifier", "fold"])
summary = summarize(fold_results)
fold_results.to_csv(OUTPUT_DIR / "fold_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
print(summary.to_string(index=False))

## Huấn luyện embedding + 1D CNN và embedding + GRU

In [ ]:
exec(TEMPORAL_SOURCE)

## Kết quả so với baseline video của bài báo

In [ ]:
import pandas as pd
from IPython.display import display

fold_results = pd.read_csv(OUTPUT_DIR / "fold_results.csv")
summary = pd.read_csv(OUTPUT_DIR / "summary.csv")
display(fold_results.round(4))
display(summary.round(4))
if RUN_MODE == "smoke":
    print("Smoke mode chỉ kiểm tra pipeline, không dùng để kết luận so sánh với bài báo.")
else:
    print("Retrained video-only CNN balanced accuracy: 89.40%")
    print("Published video-only CNN balanced accuracy: 90.66%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot = summary.copy()
values = 100 * plot["balanced_accuracy_mean"].to_numpy()
errors = 100 * (plot["balanced_accuracy_ci_high"] - plot["balanced_accuracy_mean"]).to_numpy()
positions = np.arange(len(plot))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(positions, values, yerr=errors if np.isfinite(errors).all() else None, capsize=5)
ax.axhline(89.402, color="tab:red", linestyle=":", label="Retrained video CNN: 89.40%")
ax.axhline(90.6606467, color="black", linestyle="--", label="Published video CNN: 90.66%")
ax.set_xticks(positions, plot["classifier"], rotation=15)
ax.set_ylabel("Balanced accuracy (%)")
ax.set_ylim(0, 100)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "comparison.png", dpi=180)
plt.show()

In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Artifacts:", archive)